In [1]:
# In the previous lesson we built a RAG pipeline with RAGBase.rag() and saw it fail on the "Olama" typo. The search returned nothing useful, and the LLM had no way to recover.

# The flow that broke:

In [2]:
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

In [3]:
# The difference is about who makes the decisions:

#     With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
#     With an agent, the LLM decides. It chooses which actions to take and when to stop.

# The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [4]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
MODEL = "openai/gpt-oss-20b"


In [6]:
# ASKING WITHOUT TOOLS,
message1 = [
    {"role": "user", "content": "what is olama"}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
)

response.choices[0].message.content

'**Olama** is an AI‑platform company that makes it easy for developers and businesses to build, train, and deploy large language models (LLMs) and other AI workloads.  Think of it as a “universal AI developer hub” that sits between you and the underlying models, providing a single, developer‑friendly interface while hiding the complexity of infrastructure and model‑management.\n\n---\n\n## What Olama does\n\n| Feature | What it means | Why it matters |\n|---------|---------------|----------------|\n| **Unified API** | A single REST/GraphQL endpoint you call for any supported model (e.g., GPT‑4, Llama‑2, Claude, etc.) | No need to remember different vendor URLs or auth methods. |\n| **Model Training & Fine‑tuning** | Fine‑tune existing models on your own data via a web UI or API. | Tailor the AI’s behavior to your brand voice, domain knowledge, or regulatory requirements. |\n| **Deployment & Scaling** | Run the model on Olama’s managed infrastructure or export to your own cloud. | Zero‑

In [7]:
from ingest import load_faq_data, build_index

# Load the documents and build the index
docs = load_faq_data()
index = build_index(docs)


In [8]:
# Defining the tool

# First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:


# Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
        }
    }
    
}

In [10]:
# Sending the question with the tool

# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

In [11]:
# Executing the function and sending the result back

# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

tool_calls =response.choices[0].message.tool_calls

call = tool_calls[0].function

args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [12]:
# Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

# 1. Add the assistant's message (which includes the tool call) to the history. 
# MUST be .append(), not .extend()
message1.append(response.choices[0].message)

# 2. Get the specific tool call object to retrieve its ID
tool_call = response.choices[0].message.tool_calls[0]

# 3. Append the tool's result to the history in the format OpenAI expects
message1.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # <-- The ID is here!
    "content": result_json
})


In [13]:
# Asking the model again

# We call the API a second time with the expanded history:

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
    tools=[search_tool],
)

response.choices[0].message.content

'**Olama** is a SaaS platform that turns a company’s internal data (documents, emails, chats, knowledge bases, etc.) into a searchable, AI‑powered “living knowledge base.”  \nBelow is a quick rundown of what Olama offers, why it matters, and who’s using it.\n\n| Feature | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Vector‑based search** | Embeds every piece of content into a high‑dimensional vector space and performs similarity search (cosine, dot‑product, etc.). | Finds the *most relevant* answer even if the exact wording isn’t present. |\n| **Contextual “Answer‑Engine”** | Uses an LLM (e.g., GPT‑4, Claude, etc.) to generate natural‑language responses that reference the underlying documents. | Turns raw data into a conversational assistant that can explain or summarize. |\n| **Auto‑updates** | Periodically re‑indexes new or changed content. | Keeps the knowledge base fresh without manual effort. |\n| **Fine‑tuning / Prompt‑engineering** | Lets tea

In [14]:

# Token usage and cost

# We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

# The response has a usage field with the token counts:

# usage = response.usage
# usage.input_tokens, usage.output_tokens
response.usage
completion_price = 0.75/ 1_000_000
prompt_price = 4.50 / 1_000_000

cost = [
    response.usage.completion_tokens * completion_price+
    response.usage.prompt_tokens * prompt_price

]
cost

[0.005796000000000001]

In [15]:
# For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.


def calculate_model_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_model_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [16]:
# # Conversationl history
# 1.Make a call to the LLM
# 2. LLM decided to invoke the search('params')
# 3. We invoke the search, we have the results 
# 4. send the results back to the LLM (another call)<--
# 5. LLM processes the results
# 6. LLM gives the answer

In [17]:
# The Agentic Loop

# Video: Watch this lesson

# In the previous lesson, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

# That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

In [18]:
# Anatomy of an agent

# With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

# An agent has three parts:

#     Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
#     Tools, the functions the agent can call to carry out the task. For us that's only search.
#     Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.


In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [20]:
# A function-call helper

# We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

# def make_call(call):
#     args = json.loads(call.arguments)

#     if call.name == "search":
#         result = search(**args)

#     result_json = json.dumps(result, indent=2)

#     return {
#         "type": "function_call_output",
#         "call_id": call.call_id,
#         "output": result_json,
#     }

import json

def make_call(tool_call):
    # Extract the function from the tool_call
    call = tool_call.function
    
    # Parse the arguments
    args = json.loads(call.arguments)
    
    # Run the actual search function
    result = search(**args)
    
    # Return the result as a raw JSON string
    return json.dumps(result, indent=2)


In [21]:
# Processing one response

# Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool],
)


# messages.extend(response.choices[0].message)
# has_function_calls = False

# for item in response.choices[0].message:
#     if item.type == "function_call":
#         print("function_call:", item.name, item.arguments)
#         call_output = make_call(item)
#         messages.append(call_output)
#         has_function_calls = True

#     elif item.type == "message":
#         print("ASSISTANT:")
#         print(item.content[0].text)

# 1. ALWAYS use append (with .model_dump() to be safe) to save the assistant's message
messages.append(response.choices[0].message.model_dump())

# 2. Check if the model actually made any tool calls
if response.choices[0].message.tool_calls:
    has_function_calls = True
    
    # 3. Iterate over the tool_calls list specifically
    for tool_call in response.choices[0].message.tool_calls:
        # tool_call.type will be "function"
        if tool_call.type == "function":
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            
            # Pass the actual function details to your make_call helper
            call_output = make_call(tool_call)
            
            # OpenAI requires you to pass the tool_call_id back!
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": call_output
            })
else:
    has_function_calls = False


function_call: search {"query":"join the course how to enroll"}


In [24]:
# 1. ALWAYS initialize the messages list fresh at the top of the cell!
question = "Is James enrolled in the course?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    # 2. Call the model
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[search_tool],
    )

    # 3. Clean the assistant's message for Groq compatibility
    assistant_message = response.choices[0].message
    assistant_dict = assistant_message.model_dump(exclude_none=True)
    
    # Remove unsupported keys
    assistant_dict.pop("annotations", None)
    assistant_dict.pop("audio", None)
    assistant_dict.pop("refusal", None)
    
    # Append the cleaned dictionary
    messages.append(assistant_dict)

    # 4. Check for tool calls
    if assistant_message.tool_calls:
        has_function_calls = True
        
        for tool_call in assistant_message.tool_calls:
            if tool_call.type == "function":
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                
                # Execute the tool
                call_output = make_call(tool_call)
                
                # Append the tool result
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": call_output
                })
    else:
        print("ASSISTANT:")
        print(assistant_message.content)

    it += 1
    
    # Break if we are done
    if not has_function_calls:
        break


iteration #1...
function_call: search {"query":"James enrolled course"}
iteration #2...
function_call: search {"query":"James enrollment course"}
iteration #3...
ASSISTANT:
I’m not able to find any record indicating that a student named James is currently enrolled in this course. If you have a student ID, enrollment number, or can check the class roster on the course platform, that will give you a definitive answer. Let me know if there’s anything else you’d like to look into—perhaps enrollment procedures, how to add a student, or anything else course‑related!
